# Dataset Grid Visualization

Displays a 2-row × 4-column grid for a given sample index:

| | Image | Depth | Masks overlaid | Touch point overlaid |
|---|---|---|---|---|
| **Greatest Hits** | raw frame | depth map | stick + object masks | touch region |
| **EPIC Kitchen** | raw frame | depth map | stick + object masks | touch region |

Call `show_grid(n)` with any integer index to visualise sample `n` from each dataset.

In [1]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import (
    AutoImageProcessor,
    AutoModelForDepthEstimation,
    SegGptForImageSegmentation,
    SegGptImageProcessor,
)

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from inference_script.src.touch import compute_touch_region_v2
from inference_script.src.visualization import draw_dual_mask_viz
from inference_script.src.depth import _apply_clahe, _adaptive_sharpen, _blur_score, sharpen_depth
print("Modules imported successfully.")

/home/scur0813/.conda/envs/touch_from_segmentation/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modules imported successfully.


In [2]:
print("Modules imported successfully.")

Modules imported successfully.


## Load models

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

_SEGGPT_ID = "BAAI/seggpt-vit-large"
seggpt_proc  = SegGptImageProcessor.from_pretrained(_SEGGPT_ID)
seggpt_model = SegGptForImageSegmentation.from_pretrained(_SEGGPT_ID).to(device).eval()
print(f"SegGPT on {device}")

_DEPTH_ID = "depth-anything/Depth-Anything-V2-Small-hf"
depth_proc  = AutoImageProcessor.from_pretrained(_DEPTH_ID)
depth_model = AutoModelForDepthEstimation.from_pretrained(_DEPTH_ID).to(device).eval()
print(f"DepthAnything V2 on {device}")

Loading weights: 100%|██████████| 354/354 [00:00<00:00, 1200.52it/s]


SegGPT on cpu


Loading weights: 100%|██████████| 287/287 [00:00<00:00, 4008.10it/s]

DepthAnything V2 on cpu


## Dataset paths

In [4]:
# ── Annotation roots ────────────────────────────────────────────────────────
GH_ANNOTATIONS   = PROJECT_ROOT / "data" / "greatest_hits" / "annotations"
EPIC_ANNOTATIONS = PROJECT_ROOT / "data" / "epic_kitchen"  / "annotations"

# ── Load manifests ──────────────────────────────────────────────────────────
with open(GH_ANNOTATIONS   / "train.json") as f:
    GH_SAMPLES = json.load(f)

with open(GH_ANNOTATIONS   / "train_ctx_index.json") as f:
    GH_CTX_INDEX = json.load(f)

with open(EPIC_ANNOTATIONS / "train.json") as f:
    EPIC_SAMPLES = json.load(f)

with open(EPIC_ANNOTATIONS / "train_ctx_index.json") as f:
    EPIC_CTX_INDEX = json.load(f)

print(f"Greatest Hits samples : {len(GH_SAMPLES)}")
print(f"EPIC Kitchen samples  : {len(EPIC_SAMPLES)}")

Greatest Hits samples : 24032
EPIC Kitchen samples  : 42775


## Helper utilities

In [5]:
# ── Mask / image helpers ─────────────────────────────────────────────────────

def resolve_hand_path(s: dict) -> Path | None:
    """Return the hand/stick mask path that exists on disk."""
    for key in ("hand_mask_path", "stick_mask_path"):
        val = s.get(key)
        if val and Path(val).exists():
            return Path(val)
    img_stem = Path(s.get("image_path", "")).stem
    folder   = Path(s.get("image_path", "")).parent
    for suffix in ("_stick.png", "_hand.png"):
        candidate = folder / (img_stem + suffix)
        if candidate.exists():
            return candidate
    return None


def _context_key(s: dict) -> str:
    """Return the grouping key used for context sampling."""
    raw = s.get("object_name")
    object_name = raw.strip() if isinstance(raw, str) else ""
    return object_name if object_name else s.get("video_id", "")


def overlay(img_np, mask_np, color=(0, 200, 80), alpha=0.5) -> np.ndarray:
    """Blend a binary mask over an RGB image."""
    out = img_np.astype(float)
    m = mask_np > 0
    for c, v in enumerate(color):
        out[:, :, c] = np.where(m, out[:, :, c] * (1 - alpha) + v * alpha, out[:, :, c])
    return np.clip(out, 0, 255).astype(np.uint8)


def colorize_depth(depth: np.ndarray) -> np.ndarray:
    """Map a normalised [0,1] depth array to an inferno RGB image."""
    d_min, d_max = depth.min(), depth.max()
    d_norm = (depth - d_min) / (d_max - d_min) if d_max > d_min else np.zeros_like(depth)
    return (plt.get_cmap("inferno")(d_norm)[:, :, :3] * 255).astype(np.uint8)


# ── Model inference helpers ───────────────────────────────────────────────────

def run_seggpt(target_pil, prompt_pil, prompt_mask_pil) -> np.ndarray:
    """Run SegGPT → 2-D numpy array (H, W), 0=background 1+=foreground."""
    inputs = seggpt_proc(
        images=target_pil,
        prompt_images=prompt_pil,
        prompt_masks=prompt_mask_pil,
        return_tensors="pt",
    ).to(device)
    torch.manual_seed(SEED)
    with torch.inference_mode():
        out = seggpt_model(**inputs)
    return seggpt_proc.post_process_semantic_segmentation(
        out, target_sizes=[target_pil.size[::-1]]
    )[0].cpu().numpy()


def predict_depth(pil_img, clahe_clip: float = 0.0) -> np.ndarray:
    """Run DepthAnything V2 → float32 (H, W) in [0, 1]."""
    if clahe_clip > 0.0:
        pil_img = _apply_clahe(pil_img, clip_limit=clahe_clip)
    pil_img = _adaptive_sharpen(pil_img, _blur_score(np.array(pil_img)))
    inputs = depth_proc(images=pil_img, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.inference_mode():
        raw = depth_model(**inputs).predicted_depth
    depth = F.interpolate(
        raw.unsqueeze(1), size=(pil_img.height, pil_img.width),
        mode="bicubic", align_corners=False,
    ).squeeze().cpu().numpy()
    d_min, d_max = depth.min(), depth.max()
    return (depth - d_min) / (d_max - d_min) if d_max > d_min else np.zeros_like(depth)


def run_pipeline(ctx_img, ctx_stick, ctx_obj, tgt_img, tgt_img_np):
    """Run the full SegGPT + depth + touch pipeline for one sample pair.

    Returns
    -------
    stick_mask : np.ndarray  uint8 (H, W)
    obj_mask   : np.ndarray  uint8 (H, W)
    touch_mask : np.ndarray  uint8 (H, W)
    depth      : np.ndarray  float32 (H, W) in [0, 1]
    """
    stick_pred = run_seggpt(tgt_img, ctx_img, ctx_stick)
    stick_mask = (stick_pred > 0).astype(np.uint8) * 255

    obj_pred = run_seggpt(tgt_img, ctx_img, ctx_obj)
    obj_mask = (obj_pred > 0).astype(np.uint8) * 255

    depth = predict_depth(tgt_img)
    depth = sharpen_depth(depth, tgt_img_np, radius=4, eps=0.1)

    touch_mask = compute_touch_region_v2(
        stick_mask, obj_mask, depth,
        erode_radius=0,
        dilation_radius=10,
        abs_d_threshold=0.05,
        local_radius=10,
    )
    return stick_mask, obj_mask, touch_mask, depth

## Sample loader

In [6]:
def load_sample(annotations: list, ctx_index: dict, sample_idx: int) -> dict:
    """
    Load target + context data for sample `sample_idx` in the given annotation list.

    Parameters
    ----------
    annotations : list   Full annotation list for one dataset.
    ctx_index   : dict   Pre-built context index ({key: [indices…]}).
    sample_idx  : int    Which sample to use as the *target*.

    Returns
    -------
    dict with keys: tgt_img, tgt_img_np, ctx_img, ctx_stick, ctx_obj,
                    tgt_stick_gt, tgt_obj_gt, tgt_touch_gt
    """
    tgt_sample = annotations[sample_idx % len(annotations)]

    # ── find a valid context frame ────────────────────────────────────────────
    ctx_key      = _context_key(tgt_sample)
    ctx_pool     = ctx_index.get(ctx_key, [])
    ctx_sample   = None
    for ci in ctx_pool:
        candidate = annotations[ci]
        if resolve_hand_path(candidate) is not None and ci != sample_idx % len(annotations):
            ctx_sample = candidate
            break
    if ctx_sample is None:
        raise ValueError(
            f"No valid context frame found for sample {sample_idx} "
            f"(key='{ctx_key}'). Try a different index."
        )

    hand_path = resolve_hand_path(ctx_sample)
    touch_key = "touch_mask_path" if "touch_mask_path" in tgt_sample else "target_path"

    return dict(
        tgt_img      = Image.open(tgt_sample["image_path"]).convert("RGB"),
        tgt_img_np   = np.array(Image.open(tgt_sample["image_path"]).convert("RGB")),
        ctx_img      = Image.open(ctx_sample["image_path"]).convert("RGB"),
        ctx_stick    = Image.open(hand_path).convert("L"),
        ctx_obj      = Image.open(ctx_sample["object_mask_path"]).convert("L"),
        tgt_stick_gt = Image.open(resolve_hand_path(tgt_sample)).convert("L")
                       if resolve_hand_path(tgt_sample) else None,
        tgt_obj_gt   = Image.open(tgt_sample["object_mask_path"]).convert("L"),
        tgt_touch_gt = Image.open(tgt_sample[touch_key]).convert("L")
                       if tgt_sample.get(touch_key) else None,
    )

## Main grid function

```
show_grid(n)
```

Renders a **2 × 4** grid:

| row | col 0 | col 1 | col 2 | col 3 |
|---|---|---|---|---|
| Greatest Hits | Image | Depth | Masks overlaid | Touch overlaid |
| EPIC Kitchen  | Image | Depth | Masks overlaid | Touch overlaid |

In [7]:
# ── Colours used for mask overlays ───────────────────────────────────────────
STICK_COLOR = (30,  150, 255)   # blue
OBJ_COLOR   = (255, 140,  30)   # orange
TOUCH_COLOR = (220,  40, 220)   # magenta

def save_individual_images(data: dict, filename: str) -> None:
    """
    Run the pipeline for one sample and save the four panels as separate images.

    Parameters
    ----------
    data     : dict    Output of load_sample() for one sample.
    filename : str     Base filename (without extension) to save the panels as.
    """
    img_panel, depth_panel, masks_panel, touch_panel = _build_row_panels(data)

    Image.fromarray(img_panel).save(f"{filename}_image.png")
    Image.fromarray(depth_panel).save(f"{filename}_depth.png")
    Image.fromarray(masks_panel).save(f"{filename}_masks.png")
    Image.fromarray(touch_panel).save(f"{filename}_touch.png")


def _build_row_panels(data: dict) -> tuple:
    """
    Run the full pipeline for one dataset sample and return the four panels
    that make up one grid row.

    Returns (img_panel, depth_panel, masks_panel, touch_panel) — all uint8 RGB arrays.
    """
    stick_mask, obj_mask, touch_mask, depth = run_pipeline(
        data["ctx_img"],
        data["ctx_stick"],
        data["ctx_obj"],
        data["tgt_img"],
        data["tgt_img_np"],
    )

    img_panel   = data["tgt_img_np"]
    depth_panel = colorize_depth(depth)
    masks_panel = overlay(
        overlay(data["tgt_img_np"], stick_mask, STICK_COLOR),
        obj_mask, OBJ_COLOR
    )
    # Touch: draw the touch zone; fall back to a plain mask overlay if
    # draw_dual_mask_viz is not available in this environment.
    try:
        touch_panel = draw_dual_mask_viz(
            data["tgt_img_np"], stick_mask, obj_mask, touch_mask, [], []
        )
    except Exception:
        touch_panel = overlay(data["tgt_img_np"], touch_mask, TOUCH_COLOR)
    

    return img_panel, depth_panel, masks_panel, touch_panel

def save_images(n):
    print(f"Loading Greatest Hits sample {n}…")
    gh_data   = load_sample(GH_SAMPLES,   GH_CTX_INDEX,   n)
    print(f"Loading EPIC Kitchen sample {n}…")
    epic_data = load_sample(EPIC_SAMPLES, EPIC_CTX_INDEX, n)

    print("Saving Greatest Hits panels…")
    save_individual_images(gh_data, f"{n}_gh_sample")
    print("Saving EPIC Kitchen panels…")
    save_individual_images(epic_data, f"{n}_epic_sample")


def show_grid(n: int, figsize_per_panel: tuple = (4, 3)) -> None:
    """
    Display a 2-row × 4-column visualisation grid.

    Parameters
    ----------
    n                : int    Sample index (wraps around if out of range).
    figsize_per_panel: tuple  (width, height) in inches for each panel cell.
    """
    COLS = ["Image", "Depth", "Masks overlaid", "Touch point overlaid"]
    ROWS = ["Greatest Hits", "EPIC Kitchen"]

    print(f"Loading Greatest Hits sample {n}…")
    gh_data   = load_sample(GH_SAMPLES,   GH_CTX_INDEX,   n)
    print(f"Loading EPIC Kitchen sample {n}…")
    epic_data = load_sample(EPIC_SAMPLES, EPIC_CTX_INDEX, n)

    print("Running pipeline for Greatest Hits…")
    gh_panels   = _build_row_panels(gh_data)
    print("Running pipeline for EPIC Kitchen…")
    epic_panels = _build_row_panels(epic_data)

    # ── Layout ────────────────────────────────────────────────────────────────
    n_cols, n_rows = len(COLS), len(ROWS)
    fw = figsize_per_panel[0] * n_cols
    fh = figsize_per_panel[1] * n_rows

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(fw, fh),
        squeeze=False,
        gridspec_kw={"hspace": 0.06, "wspace": 0.04},
    )

    all_panels = [gh_panels, epic_panels]

    for r, (row_panels, row_label) in enumerate(zip(all_panels, ROWS)):
        for c, (panel, col_label) in enumerate(zip(row_panels, COLS)):
            ax = axes[r][c]
            ax.imshow(panel)
            ax.axis("off")

            # Column headers (top row only)
            if r == 0:
                ax.set_title(col_label, fontsize=11, fontweight="bold", pad=6)

            # Row labels (left column only)
            if c == 0:
                ax.set_ylabel(
                    row_label,
                    fontsize=11, fontweight="bold",
                    rotation=90, labelpad=8,
                )
                ax.yaxis.set_label_position("left")
                ax.tick_params(left=False, labelleft=False)

    # ── Legend ────────────────────────────────────────────────────────────────
    legend_patches = [
        mpatches.Patch(color=np.array(STICK_COLOR) / 255, label="Stick/hand mask"),
        mpatches.Patch(color=np.array(OBJ_COLOR)   / 255, label="Object mask"),
        mpatches.Patch(color=np.array(TOUCH_COLOR) / 255, label="Touch region"),
    ]
    fig.legend(
        handles=legend_patches,
        loc="lower center",
        ncol=3,
        fontsize=9,
        framealpha=0.85,
        bbox_to_anchor=(0.5, -0.02),
    )

    fig.suptitle(
        f"Dataset grid — sample index {n}",
        fontsize=13, fontweight="bold", y=1.01,
    )
    plt.tight_layout()
    plt.show()

## Run it!

Change the integer argument to browse different samples.

In [ ]:
save_images(0)
save_images(3)

Loading Greatest Hits sample 0…
Loading EPIC Kitchen sample 0…
Saving Greatest Hits panels…
Saving EPIC Kitchen panels…
Loading Greatest Hits sample 3…
Loading EPIC Kitchen sample 3…
Saving Greatest Hits panels…
Saving EPIC Kitchen panels…


: 